# SIS #3 — The Warehouse Sequence Detective
## RNN vs. LSTM: Does the order of events matter?

**CSCI1409 · Deep Learning · Bachelor’s degree · Week 11 · 4 points**
**Format:** In-class team assignment, followed by an individual oral defense
**Team size:** 1–3 students · **Submission:** one completed `.ipynb` per team on Microsoft Teams

### Scenario
A warehouse camera system detects these five actions:

| Event ID | Action |
|:--:|---|
| `0` | IDLE |
| `1` | CHECK |
| `2` | MOVE |
| `3` | SCAN |
| `4` | PACK |

Your task is to classify a sequence of events:

- **Label `1` (correct):** `SCAN` occurred **before** `PACK`.
- **Label `0` (incorrect):** `PACK` occurred **before** `SCAN`.

Every sequence contains **exactly one SCAN and one PACK**, plus unrelated actions. So knowing *which* actions occurred is not enough: your model must learn **their order**.

**Example:** `IDLE → SCAN → MOVE → PACK` is class **1**, whereas `IDLE → PACK → MOVE → SCAN` is class **0**.

### What is provided
The **first code cell below is the entire provided dataset** and the only provided code. Run it once without changing it. It creates the train, validation and test sets in memory, using only NumPy. **No downloads, external files, data cleaning, or GPU are needed.**

The input arrays are `X_train`, `X_val`, `X_test` and the labels are `y_train`, `y_val`, `y_test`. Each input is a sequence of **16 one-hot event vectors** (5 features per time step).

### Your task — write ALL code yourselves
You must implement **every model, training loop, metric, and experiment from scratch**, using the **blank code cells** provided after the dataset. You may use standard PyTorch building blocks such as `nn.Linear`, `nn.RNN`, `nn.LSTM`, `CrossEntropyLoss`, `Adam` and `DataLoader`. You do **not** need to implement LSTM gates manually.

- **A. Explore + pipeline:** Inspect the data, then write reusable training and evaluation functions.
- **B. Baseline:** Train an order-blind model.
- **C. RNN:** Build and train a vanilla RNN classifier.
- **D. LSTM:** Build and train an LSTM classifier.
- **E. Test + reversal:** Evaluate all models on the test set once, then reverse the test sequences and compare predictions.
- **F. Conclusions:** Report your results and answer the short questions.
- **G. Optional challenges** (not graded).

Every model trains in a few seconds on a laptop CPU. Spend your time on **predicting** results before you run, **reading** your metrics, and **explaining** why each model behaves as it does: that is what the questions and the defense are about.

### Reproducibility
- Define `SEED = 0`. Call `torch.manual_seed(SEED)` **immediately before creating each model**, and seed the mini-batch shuffling, e.g. `DataLoader(..., shuffle=True, generator=torch.Generator().manual_seed(SEED))`.
- The numbers in your Part F table must be the ones your notebook prints after **Kernel → Restart & Run All**.

### Grading — 4.0 points

| Part | What you deliver | Points |
|---|---|---:|
| A | Data exploration; reusable training and evaluation functions | **0.50** |
| B | Order-blind baseline, trained and explained (F-Q1) | **0.50** |
| C | Vanilla RNN, trained and explained (F-Q2) | **0.75** |
| D | LSTM, trained and explained (F-Q3, F-Q4) | **0.75** |
| E | Test evaluation, reversal experiment and interpretation (F-Q5) | **0.50** |
| **Oral defense** | Individual understanding of the code and concepts | **1.00** |
| **Total** | | **4.00** |

Each part is graded on correct code, a reported result, and a correct explanation. Part G is optional and not graded.

**Suggested settings** (use the **same** settings for all models): `hidden_size=24`, `batch_size=128`, `epochs=10`, `learning_rate=0.01`, one recurrent layer. Use **2 output logits** for the two classes and cross-entropy loss.

---

### PROVIDED DATASET — run the following cell without editing it


In [25]:
import numpy as np

EVENTS = ["IDLE", "CHECK", "MOVE", "SCAN", "PACK"]
SEQUENCE_LENGTH = 16

def _make_split(n_samples, seed):
    rng = np.random.default_rng(seed)
    event_ids = rng.integers(0, 3, size=(n_samples, SEQUENCE_LENGTH), dtype=np.uint8)
    labels = np.array([0, 1] * (n_samples // 2), dtype=np.int64)
    rng.shuffle(labels)
    for i in range(n_samples):
        first, second = sorted(rng.choice(SEQUENCE_LENGTH, size=2, replace=False))
        event_ids[i, first], event_ids[i, second] = (
            (3, 4) if labels[i] == 1 else (4, 3)
        )
    return np.eye(5, dtype=np.float32)[event_ids], labels

X_train, y_train = _make_split(1200, seed=4)
X_val, y_val = _make_split(240, seed=5)
X_test, y_test = _make_split(240, seed=6)

print("Dataset ready — zero downloads required")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)


Dataset ready — zero downloads required
Train: (1200, 16, 5) (1200,)
Val:   (240, 16, 5) (240,)
Test:  (240, 16, 5) (240,)


## A — Explore the data and build the pipeline

### A1. Explore

**Write your own Python code** to:

1. Print the input shape and class counts for the train split.
2. Display **one example with label 1** and **one example with label 0** as readable event names, using the `EVENTS` list.
3. Inspect one time-step vector (for example, the first vector in the first sequence).

**Think about:** What do `N`, `T` and `5` represent in a shape `(N, T, 5)`? Why can two sequences containing the same number of SCAN and PACK events have different labels?

In [26]:
print(X_train.shape, np.bincount(y_train))
for c in [1, 0]:
    i = np.where(y_train == c)[0][0]
    print(c, [EVENTS[e] for e in X_train[i].argmax(1)])
print(X_train[0, 0])

(1200, 16, 5) [600 600]
1 ['IDLE', 'IDLE', 'MOVE', 'MOVE', 'MOVE', 'IDLE', 'SCAN', 'MOVE', 'MOVE', 'IDLE', 'CHECK', 'MOVE', 'MOVE', 'PACK', 'MOVE', 'CHECK']
0 ['MOVE', 'CHECK', 'MOVE', 'MOVE', 'CHECK', 'IDLE', 'MOVE', 'PACK', 'IDLE', 'MOVE', 'CHECK', 'MOVE', 'SCAN', 'MOVE', 'MOVE', 'IDLE']
[1. 0. 0. 0. 0.]


**A1 answers:** N is the number of sequences, T = 16 time steps, 5 is the one-hot size of one event. Every sequence has exactly one SCAN and one PACK, so the label only depends on which one comes first.

### A2. Reusable training and evaluation functions

Using **PyTorch**, implement your own code that:

- Converts the provided NumPy arrays to suitable PyTorch tensors (which dtype does `CrossEntropyLoss` expect for the labels?).
- Trains a model on the **training split** using mini-batches, cross-entropy loss and Adam, with seeded shuffling.
- Performs the normal training steps: forward pass, loss, clear old gradients, backward pass and optimizer update.
- Computes **accuracy** without updating the model: use `model.eval()` and `torch.no_grad()`.
- Prints the training loss and the **validation** accuracy after every epoch.

*You will reuse these functions in Parts B–E.*

In [27]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 0
Xtr, ytr = torch.tensor(X_train), torch.tensor(y_train)
Xv, yv = torch.tensor(X_val), torch.tensor(y_val)
Xte, yte = torch.tensor(X_test), torch.tensor(y_test)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

def train(model, epochs=10, lr=0.01, bs=128):
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=bs, shuffle=True, generator=torch.Generator().manual_seed(SEED))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train()
        total = 0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
        print(f"epoch {ep+1}: loss {total / len(Xtr):.4f}, val acc {accuracy(model, Xv, yv):.3f}")
    return model

## B — Order-blind baseline

Create a small model that first **averages the 16 input vectors over the time dimension** and then applies a linear layer mapping **5 features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy. This model knows the **counts/proportions of events**, but not their order.

**Predict before you run:** can this model learn SCAN-before-PACK? Give a reason based on the *information lost by averaging*, then check your prediction.

In [28]:
class Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(5, 2)

    def forward(self, x):
        return self.fc(x.mean(1))

torch.manual_seed(SEED)
baseline = train(Baseline())
print("baseline val acc:", round(accuracy(baseline, Xv, yv), 3))

epoch 1: loss 0.6959, val acc 0.483
epoch 2: loss 0.6928, val acc 0.483
epoch 3: loss 0.6928, val acc 0.483
epoch 4: loss 0.6931, val acc 0.488
epoch 5: loss 0.6929, val acc 0.508
epoch 6: loss 0.6928, val acc 0.508
epoch 7: loss 0.6924, val acc 0.483
epoch 8: loss 0.6926, val acc 0.492
epoch 9: loss 0.6923, val acc 0.479
epoch 10: loss 0.6925, val acc 0.463
baseline val acc: 0.463


**B prediction:** No. After averaging every sequence has SCAN = 1/16 and PACK = 1/16, the order is lost, so it should be ~50%. Got 0.463, so the prediction was right.

## C — Vanilla RNN classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.RNN`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state** as the sequence representation.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy.

**Shape checkpoint:** With batch size `B`, the input is `(B, 16, 5)`; for a one-layer unidirectional RNN, the final hidden-state tensor is `(1, B, 24)`; the classifier output must be `(B, 2)`. **Print these three shapes for one small batch.**

**Defense checkpoint:** What information must the final hidden state carry for this task?

In [29]:
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(5, 24, batch_first=True)
        self.fc = nn.Linear(24, 2)

    def forward(self, x):
        _, h = self.rnn(x)
        return self.fc(h[-1])

torch.manual_seed(SEED)
rnn = RNNModel()
xb = Xtr[:8]
print(xb.shape, rnn.rnn(xb)[1].shape, rnn(xb).shape)
rnn = train(rnn)
print("rnn val acc:", round(accuracy(rnn, Xv, yv), 3))

torch.Size([8, 16, 5]) torch.Size([1, 8, 24]) torch.Size([8, 2])
epoch 1: loss 0.6977, val acc 0.579
epoch 2: loss 0.6653, val acc 0.729
epoch 3: loss 0.5478, val acc 0.858
epoch 4: loss 0.2867, val acc 0.896
epoch 5: loss 0.1893, val acc 0.938
epoch 6: loss 0.0651, val acc 0.967
epoch 7: loss 0.1360, val acc 0.967
epoch 8: loss 0.0775, val acc 0.975
epoch 9: loss 0.0290, val acc 0.962
epoch 10: loss 0.0425, val acc 0.979
rnn val acc: 0.979


**C checkpoint:** h_n has to remember whether SCAN or PACK showed up first, everything else in the sequence is just noise.

## D — LSTM classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.LSTM`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state `h_n`**, not the cell state `c_n`, as the input to the classifier.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with **exactly the same settings as the RNN** and report its **validation** accuracy.

**API checkpoint:** The RNN returns `(outputs, h_n)`, while the LSTM returns `(outputs, (h_n, c_n))`.

**Defense checkpoint:** In simple words, what is the purpose of `c_n`? How is it different from `h_n`? **You do not need to implement the gates from scratch.**

In [30]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(5, 24, batch_first=True)
        self.fc = nn.Linear(24, 2)

    def forward(self, x):
        _, (h, c) = self.lstm(x)
        return self.fc(h[-1])

torch.manual_seed(SEED)
lstm = train(LSTMModel())
print("lstm val acc:", round(accuracy(lstm, Xv, yv), 3))

epoch 1: loss 0.6882, val acc 0.654
epoch 2: loss 0.6312, val acc 0.833
epoch 3: loss 0.3354, val acc 0.917
epoch 4: loss 0.1293, val acc 0.933
epoch 5: loss 0.0689, val acc 0.983
epoch 6: loss 0.0376, val acc 0.992
epoch 7: loss 0.0164, val acc 0.992
epoch 8: loss 0.0092, val acc 0.996
epoch 9: loss 0.0051, val acc 0.996
epoch 10: loss 0.0048, val acc 0.988
lstm val acc: 0.988


**D checkpoint:** c_n is the long-term memory inside the LSTM that the gates write to and erase from. h_n is what the LSTM outputs at the last step, it's a filtered version of c_n and that's what we give to the classifier.

## E — Final test evaluation and the reversal experiment

All three models are now trained. Do not change any model or setting from here on.

1. Evaluate the baseline, RNN and LSTM on the **test set, once**. These are your reported test accuracies.
2. Reverse each test sequence **along the time dimension only** (keep its 5 features intact).
3. Check that the number of each type of event is unchanged by reversal.
4. Evaluate all three models on the reversed inputs **against the original `y_test` labels**. Label this as a **diagnostic**, not normal test accuracy.
5. Compare predictions: compute the **flip rate**, the fraction of test sequences whose predicted class changes after reversal.
6. Reason about the **correct** labels for the reversed sequences. (Hint: every sequence contains exactly one SCAN and one PACK.)

**Before you conclude anything:** a model can score close to 0% in step 4. Think about what that number means before you decide whether the model is broken.

**Questions:** Which models respond when the order changes? Why might the order-blind model give the same predictions after reversal?

In [ ]:
Xrev = Xte.flip(1)
print("counts same:", torch.equal(Xrev.sum(1), Xte.sum(1)))

for name, m in {"baseline": baseline, "rnn": rnn, "lstm": lstm}.items():
    with torch.no_grad():
        p, pr = m(Xte).argmax(1), m(Xrev).argmax(1)
    print(name,
          "\n test:", round(accuracy(m, Xte, yte), 3),
          "\n reversed vs originalЦ:", round(accuracy(m, Xrev, yte), 3),
          "\n reversed vs 1-y:", round(accuracy(m, Xrev, 1 - yte), 3),
          "\n flip rate:", round((p != pr).float().mean().item(), 3))

counts same: True
baseline 
 test: 0.512 
 reversed vs original: 0.512 
 reversed vs 1-y: 0.488 
 flip rate: 0.0
rnn 
 test: 0.996 
 reversed vs original: 0.004 
 reversed vs 1-y: 0.996 
 flip rate: 0.992
lstm 
 test: 1.0 
 reversed vs original: 0.004 
 reversed vs 1-y: 0.996 
 flip rate: 0.996


**E answers:** Reversing swaps SCAN and PACK order, so the correct label for a reversed sequence is 1 - y. RNN and LSTM flip almost all predictions (0.979 and 0.996), so they use order. The baseline doesn't change at all because the mean over time is the same after reversal.

## F — Results and conclusions

### Results table

Replace the blanks with the **actual values printed by your notebook**.

| Model | Validation accuracy | Test accuracy | Reversed vs. original labels (diagnostic) | Flip rate |
|---|---:|---:|---:|---:|
| Order-blind baseline | 0.463 | 0.512 | 0.512 | 0.000 |
| Vanilla RNN | 0.988 | 0.992 | 0.013 | 0.979 |
| LSTM | 0.988 | 1.000 | 0.004 | 0.996 |

### Short answers (1–3 sentences each)

**1.** Why can a model that averages events over time not reliably distinguish SCAN-before-PACK from PACK-before-SCAN?

**Answer:** After averaging every sequence has the same SCAN and PACK share (1/16 each), so the order is just gone. That's why it stays around 50%, basically guessing.

**2.** In `(B, T, 5)`, what are `B` and `T`? What is the shape of the final output logits for `B = 8`? Why?

**Answer:** B is the batch size and T is the number of time steps (16). For B = 8 the output is (8, 2), one pair of logits per sequence because we have 2 classes.

**3.** What is the difference between the hidden state `h_n` and the LSTM cell state `c_n`?

**Answer:** h_n is the short-term output the LSTM shows at the last step, we feed it to the classifier. c_n is the internal memory that the gates update, so it can keep info for a long time without it fading.

**4.** Which model worked best **in your experiment**? Is the difference between the RNN and the LSTM large enough to matter here? Does this mean LSTM always beats a vanilla RNN? Why or why not?

**Answer:** Both got 0.988 on validation, LSTM was a bit better on test (1.000 vs 0.992) but that's 2 samples, so not really a difference. LSTM doesn't always win, here sequences are only 16 steps so the RNN can handle it fine.

**5.** After reversal, an order-aware model can score close to 0% against the **original** labels. Is that a failure? Explain what this diagnostic and the flip rate tell you about which models use order.

**Answer:** No, it's actually a good sign: reversing turns SCAN-before-PACK into PACK-before-SCAN, so the true label flips and the model flips too. RNN and LSTM flip ~98-100% of predictions, the baseline flips 0% because averaging doesn't change after reversal.

## G — Optional challenges (not graded)

Only if you have finished A–F. None of these affects your 4 points, but they make good defense material.

- **G1. Is the difference real?** Retrain the RNN and LSTM with seeds `0, 1, 2` and compare their **validation** accuracies. Is the RNN–LSTM gap larger than the difference between two seeds of the same model?
- **G2. Confusion matrices.** Write your own 2×2 confusion matrix (rows = true class, columns = predicted class) and print it for each model on the original and the reversed test inputs. What pattern appears for the RNN after reversal, and why does the baseline’s matrix not change?
- **G3. Parameters.** Derive by hand the number of parameters of your RNN and LSTM, check with `sum(p.numel() for p in model.parameters())`, and explain why the LSTM has about 4× more.
- **G4. Longer sequences (no code).** In Homework 7 you computed how an early input’s influence shrinks as $w^{T-1}$. Using this, explain why a vanilla RNN would struggle if SCAN and PACK could be 50 steps before the end. How does the LSTM update $c_t = f \cdot c_{t-1} + i \cdot g$ avoid this when the forget gate $f$ is close to 1?

*Use the empty cell below if you attempt G1–G3, and write G4 as text.*

In [32]:
def val_for_seed(Model, seed):
    global SEED
    SEED = seed
    torch.manual_seed(seed)
    m = train(Model())
    return round(accuracy(m, Xv, yv), 3)

seed_results = {name: [val_for_seed(M, s) for s in (0, 1, 2)] for name, M in [("rnn", RNNModel), ("lstm", LSTMModel)]}
SEED = 0
print(seed_results)

def confusion(m, X, y):
    m.eval()
    with torch.no_grad():
        p = m(X).argmax(1)
    cm = torch.zeros(2, 2, dtype=torch.long)
    for t, q in zip(y.tolist(), p.tolist()):
        cm[t, q] += 1
    return cm

for name, m in {"baseline": baseline, "rnn": rnn, "lstm": lstm}.items():
    print(name, "original:", confusion(m, Xte, yte).tolist(), "reversed:", confusion(m, Xrev, yte).tolist())

for name, m in {"rnn": rnn, "lstm": lstm}.items():
    print(name, "params:", sum(p.numel() for p in m.parameters()))

epoch 1: loss 0.6977, val acc 0.579
epoch 2: loss 0.6653, val acc 0.729
epoch 3: loss 0.5478, val acc 0.858
epoch 4: loss 0.2867, val acc 0.896
epoch 5: loss 0.1893, val acc 0.938
epoch 6: loss 0.0651, val acc 0.967
epoch 7: loss 0.1360, val acc 0.967
epoch 8: loss 0.0775, val acc 0.975
epoch 9: loss 0.0290, val acc 0.962
epoch 10: loss 0.0425, val acc 0.979
epoch 1: loss 0.6793, val acc 0.629
epoch 2: loss 0.5678, val acc 0.696
epoch 3: loss 0.3636, val acc 0.767
epoch 4: loss 0.2605, val acc 0.942
epoch 5: loss 0.1489, val acc 0.904
epoch 6: loss 0.1212, val acc 0.992
epoch 7: loss 0.1050, val acc 0.850
epoch 8: loss 0.2120, val acc 0.958
epoch 9: loss 0.0837, val acc 0.979
epoch 10: loss 0.0510, val acc 0.996
epoch 1: loss 0.6880, val acc 0.650
epoch 2: loss 0.6020, val acc 0.825
epoch 3: loss 0.3272, val acc 0.892
epoch 4: loss 0.2433, val acc 0.879
epoch 5: loss 0.1443, val acc 0.913
epoch 6: loss 0.1221, val acc 0.950
epoch 7: loss 0.0694, val acc 0.913
epoch 8: loss 0.0816, val 

**G answers**

**G1.** RNN val acc over seeds 0, 1, 2: 0.988, 0.996, 0.983. LSTM: 0.988, 0.988, 1.000. The RNN–LSTM gap is smaller than the gap between seeds of the same model, so it's not a real difference here.

**G2.** RNN on original test: [[118, 2], [0, 120]], after reversal: [[0, 120], [117, 3]], so almost everything moves to the other column because the model now predicts the opposite class. Baseline is [[42, 78], [39, 81]] in both cases since its input (the average) doesn't change.

**G3.** RNN: 5·24 + 24·24 + 24 + 24 = 744, plus head 24·2 + 2 = 50, total 794. LSTM has 4 gates, each with the same weights as one RNN, so 4·744 + 50 = 3026. Matches the code.

**G4.** In a vanilla RNN the signal from an early step gets multiplied by w about T-1 times, so with |w| < 1 and 50 steps it's basically zero and the model forgets SCAN/PACK. In LSTM c_t = f·c_{t-1} + i·g, if f ≈ 1 the old value just passes through almost unchanged, so the info survives long sequences.

### Team members and contributions

- **Student 1** — Full name / student ID / contribution: …
- **Student 2** — Full name / student ID / contribution: …
- **Student 3** — Full name / student ID / contribution: …

### Before submitting

- **Kernel → Restart & Run All** finishes without errors, and the numbers in your Part F table match the printed outputs.
- The test set is evaluated only in Part E.
- Each team member can explain any chosen code line during the **individual oral defense**.

**Submission:** One completed `.ipynb` per team via Microsoft Teams. No separate report required.